# Real vs. Fake Face Classification (HOG + colour features)

## 1. Install dependencies

In [ ]:
%pip install -q scikit-image scikit-learn pandas numpy matplotlib requests pillow joblib tqdm

## 2. Imports and settings

In [ ]:
import re, time, random, warnings
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib
from joblib import Parallel, delayed
from PIL import Image, ImageOps
from skimage.feature import hog

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
)

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CSV_PATH = Path("AI_Classification-Project.csv")
IMG_DIR = Path("image_cache")
IMG_DIR.mkdir(exist_ok=True)

IMG_SIZE = 128
HOG_CELL = 8
HOG_BLOCK = 2
HOG_BINS = 9
N_WORKERS = 8
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120 Safari/537.36"}

## 3. Load and validate the CSV

In [ ]:
df = pd.read_csv(CSV_PATH)

required = {"image_id", "image_url", "label_numeric", "dataset_split"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

df["label_numeric"] = pd.to_numeric(df["label_numeric"], errors="coerce")
df = df[df["label_numeric"].isin([0, 1])].copy()
df["label_numeric"] = df["label_numeric"].astype(int)
df["host"] = df["image_url"].str.extract(r"https?://([^/]+)")[0]
df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())
display(pd.crosstab(df["dataset_split"], df["label_numeric"], margins=True))

## 4. Check for metadata leakage

Columns such as `source`, `fake_method`, `resolution`, `detection_difficulty` and `category` separate the classes perfectly, and `gender` does so largely. They are not used as features, so the model learns from image content only.

In [ ]:
for col in ["source", "fake_method", "resolution", "detection_difficulty", "category", "gender", "host"]:
    if col in df.columns:
        print(f"--- {col}")
        display(df.groupby(col, dropna=False)["label_numeric"].agg(["count", "mean"]).sort_values("count", ascending=False))

## 5. Download and cache images

In [ ]:
def light_url(url):
    return re.sub(r"([?&])w=1080", r"\1w=256", str(url))

def candidate_urls(url):
    first = light_url(url)
    urls = [first]
    if re.search(r"\.jpg($|\?)", first):
        urls.append(re.sub(r"\.jpg($|\?)", r".png\1", first))
    return urls

def cache_path(image_id):
    safe_id = re.sub(r"[^A-Za-z0-9_.-]", "_", str(image_id))
    return IMG_DIR / f"{safe_id}.jpg"

session = requests.Session()

def download_one(image_id, url, retries=3):
    out = cache_path(image_id)
    if out.exists() and out.stat().st_size > 0:
        return image_id, True
    for attempt in range(retries):
        for candidate in candidate_urls(url):
            try:
                response = session.get(candidate, timeout=25, headers=HEADERS)
                response.raise_for_status()
                with Image.open(BytesIO(response.content)) as im:
                    im = ImageOps.fit(im.convert("RGB"), (IMG_SIZE, IMG_SIZE), method=Image.BILINEAR)
                    im.save(out, quality=95)
                return image_id, True
            except Exception:
                continue
        time.sleep(2 * (attempt + 1))
    return image_id, False

todo = [(i, u) for i, u in zip(df["image_id"], df["image_url"])
        if not (cache_path(i).exists() and cache_path(i).stat().st_size > 0)]
print(f"{len(df) - len(todo)} cached; {len(todo)} to download")

failed = []
if todo:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = [pool.submit(download_one, i, u) for i, u in todo]
        for future in tqdm(as_completed(futures), total=len(futures)):
            image_id, ok = future.result()
            if not ok:
                failed.append(image_id)

print("Failed downloads:", len(failed))
if failed:
    print(df[df["image_id"].isin(failed)]["host"].value_counts().to_string())
    print("Re-run this cell to retry the failed ones.")

## 6. Extract HOG and colour features

In [ ]:
def extract_features(path, img_size=IMG_SIZE):
    try:
        with Image.open(path) as im:
            im = ImageOps.fit(im.convert("RGB"), (img_size, img_size), method=Image.BILINEAR)
            gray = np.asarray(im.convert("L"), dtype=np.float32) / 255.0
            hsv = np.asarray(im.convert("HSV")).reshape(-1, 3)
    except Exception:
        return None

    hog_features = hog(
        gray,
        orientations=HOG_BINS,
        pixels_per_cell=(HOG_CELL, HOG_CELL),
        cells_per_block=(HOG_BLOCK, HOG_BLOCK),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype(np.float32)

    color_hist, _ = np.histogramdd(hsv, bins=(8, 8, 8), range=((0, 256),) * 3)
    color_hist = color_hist.ravel().astype(np.float32)
    color_hist /= color_hist.sum() + 1e-7

    return np.concatenate([hog_features, color_hist])


paths = [cache_path(i) for i in df["image_id"]]
features = Parallel(n_jobs=-1, batch_size=32)(delayed(extract_features)(p) for p in tqdm(paths))

keep = np.array([f is not None for f in features])
if not keep.any():
    raise ValueError("No readable cached images found. Run the download cell first.")

df_img = df[keep].reset_index(drop=True)
X = np.vstack([f for f in features if f is not None])
y = df_img["label_numeric"].to_numpy()

print("Usable images:", len(df_img), "| skipped:", int((~keep).sum()))
print("Feature matrix:", X.shape)
print("Labels (0=FAKE, 1=REAL):", pd.Series(y).value_counts().to_dict())

## 7. Train / validation / test sets

In [ ]:
split = df_img["dataset_split"].astype(str).str.lower().str.strip()
train_mask = split.eq("train").to_numpy()
val_mask = split.isin(["val", "valid", "validation"]).to_numpy()
test_mask = split.eq("test").to_numpy()

if not (train_mask.any() and val_mask.any() and test_mask.any()):
    raise ValueError("dataset_split must contain train, val and test rows.")

print(f"Train={train_mask.sum()} | Validation={val_mask.sum()} | Test={test_mask.sum()}")

## 8. Train models (best model chosen on validation AUC)

In [ ]:
def metrics_row(name, y_true, probability):
    pred = (probability >= 0.5).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "precision_REAL": precision_score(y_true, pred, zero_division=0),
        "recall_REAL": recall_score(y_true, pred, zero_division=0),
        "f1_REAL": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
    }

models = {
    "HOG + HSV / SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=200, random_state=SEED)),
        ("classifier", SVC(C=2.0, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED)),
    ]),
    "HOG + HSV / ExtraTrees": Pipeline([
        ("classifier", ExtraTreesClassifier(n_estimators=250, class_weight="balanced", min_samples_leaf=2,
                                            n_jobs=-1, random_state=SEED)),
    ]),
    "HOG + HSV / RandomForest": Pipeline([
        ("classifier", RandomForestClassifier(n_estimators=250, class_weight="balanced", min_samples_leaf=2,
                                              n_jobs=-1, random_state=SEED)),
    ]),
}

def prob_real(model, features):
    return model.predict_proba(features)[:, list(model.classes_).index(1)]

results, fitted_models, test_probabilities, val_auc = [], {}, {}, {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X[train_mask], y[train_mask])
    fitted_models[name] = model
    val_auc[name] = roc_auc_score(y[val_mask], prob_real(model, X[val_mask]))
    test_probabilities[name] = prob_real(model, X[test_mask])
    results.append({**metrics_row(name, y[test_mask], test_probabilities[name]), "val_roc_auc": val_auc[name]})
    print(f"{name} done in {time.time() - t0:.0f}s")

results_df = pd.DataFrame(results).set_index("model").sort_values("val_roc_auc", ascending=False)
display(results_df.round(4))

## 9. Evaluate the best model

In [ ]:
best_name = max(val_auc, key=val_auc.get)
best_model = fitted_models[best_name]
best_prob = test_probabilities[best_name]
y_test = y[test_mask]
y_pred = (best_prob >= 0.5).astype(int)

print("Selected model (by validation AUC):", best_name)
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["FAKE", "REAL"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred, labels=[0, 1]), display_labels=["FAKE", "REAL"]
).plot(ax=axes[0], values_format="d", colorbar=False)
axes[0].set_title(f"Test confusion matrix: {best_name}")

for name, prob in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("ROC curves on held-out test set")
plt.tight_layout()
plt.show()

## 10. Inspect errors by image host

In [ ]:
test_df = df_img.loc[test_mask].copy()
test_df["prob_real"] = best_prob
test_df["pred_numeric"] = y_pred
test_df["correct"] = test_df["pred_numeric"] == test_df["label_numeric"]

display(test_df.groupby(["host", "label_numeric"]).agg(
    count=("correct", "size"), accuracy=("correct", "mean")
).round(3))

wrong = test_df.loc[~test_df["correct"]].copy()
wrong["confidence_in_prediction"] = np.where(wrong["pred_numeric"] == 1, wrong["prob_real"], 1 - wrong["prob_real"])
display(wrong.sort_values("confidence_in_prediction", ascending=False)
        [["image_id", "host", "label_numeric", "pred_numeric", "prob_real", "confidence_in_prediction"]].head(15))

## 11. Save the fitted model

In [ ]:
OUT_DIR = Path("saved_model")
OUT_DIR.mkdir(exist_ok=True)

joblib.dump({
    "model": best_model,
    "model_name": best_name,
    "img_size": IMG_SIZE,
    "label_map": {0: "FAKE", 1: "REAL"},
}, OUT_DIR / "real_fake_hog.joblib")

print("Saved:", (OUT_DIR / "real_fake_hog.joblib").resolve())

## 12. Predict a new image

In [ ]:
def predict_image(image_path, threshold=0.5):
    artifact = joblib.load(OUT_DIR / "real_fake_hog.joblib")
    feats = extract_features(image_path, artifact["img_size"])
    if feats is None:
        raise ValueError(f"Could not open image: {image_path}")

    p_real = float(prob_real(artifact["model"], feats.reshape(1, -1))[0])
    return {
        "prediction": "REAL" if p_real >= threshold else "FAKE",
        "probability_REAL": round(p_real, 4),
        "probability_FAKE": round(1 - p_real, 4),
    }

sample = test_df.iloc[0]
print("true label:", sample["label"] if "label" in test_df.columns else sample["label_numeric"])
print(predict_image(cache_path(sample["image_id"])))